# Experiment 2A · Readmission explanation infrastructure

H1 compares attribution faithfulness between A (baseline) and B (SHAP-grounded).
H2 compares B's original draft with C (verification, optional revision, and fallback).
This notebook exercises the workflow on fabricated cases with known attribution references.

## Environment and configuration

Select the HRP Python kernel created by `uv sync --locked`. The default
workflow uses scripted responses. `Ollama.discover()` prepares the local
Qwen3 adapter for a separate fictional smoke run.

In [1]:
import json
from pathlib import Path
import sys
import tempfile

from IPython.display import display

source = next(path for path in (Path("src"), Path("."), Path("HRP/src"))
              if (path / "llm_runner.py").is_file())
sys.path.insert(0, str(source.resolve()))

from experiment2_scoring import score_experiment
from experiment2_smoke import fictional_cases, run_smoke
from llm_runner import ModelConfig, Ollama, run_experiment
from llm_verification import RESPONSE_SCHEMA, build_prompt

ModelConfig()

/Users/sghosh/Documents/Github/AI380H/HRP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModelConfig(model='qwen3:8b', endpoint='http://127.0.0.1:11434', digest=None, temperature=0, seed=42, num_ctx=4096, num_predict=512)

## Explanation schema

All conditions return an explanation and up to three ordered drivers.
Driver directions are `positive` or `negative`. Sentence-count
violations are recorded in the parse trace.

In [2]:
display(RESPONSE_SCHEMA)

{'explanation': 'Nonempty string, at most 90 words. Aim for 2–3 sentences.',
 'drivers': [{'feature': 'supplied feature key',
   'direction': 'positive or negative'}]}

## Prompt templates

A sees the same feature values, units, and probability as B.
B additionally sees the SHAP reference. C begins from B's raw text.

In [3]:
example = fictional_cases()[0]
for condition in ("A", "B"):
    print(f"Condition {condition}\n{build_prompt(example, condition)}\n")

Condition A
Explain which available features most strongly influence the fitted model's readmission prediction.
State the supplied predicted probability as a percentage. Use 2–3 sentences and at most 90 words.
Describe model behavior. Avoid clinical diagnosis, causal claims, treatment advice, and invented measurements.
Use explicit feature names and direct statements about increases or decreases in the model's predicted risk.
You may express uncertainty and list fewer drivers, including none when the evidence is insufficient.
Return only a JSON object with exactly two keys: "explanation" (a string) and "drivers" (an array of 0–3 objects).
Each driver must have exactly "feature" (a supplied feature key) and "direction" ("positive" or "negative").
List drivers from most to least influential and use each feature at most once.
Attributions are unavailable here. Acknowledge uncertainty when you cannot infer model behavior.

Input JSON:
{"features": [{"name": "age", "label": "Age", "unit": "

## Fictional smoke-test setup

The fixture generator checks additivity against an independent linear
margin. Scripted responses exercise ranked attributions, abstention,
malformed JSON, a missing lab, contradictory prose, and revision failure.

In [4]:
output_directory = Path(tempfile.mkdtemp(prefix="hrp-experiment2-fictional-"))

## Runner invocation

The smoke workflow checkpoints at a four-call budget, then resumes
and verifies that a completed rerun uses zero additional calls.

In [5]:
report = run_smoke(output_directory)
display(report)

{'cases': 8,
 'outputs': {'A': 8, 'B': 8, 'C': 8},
 'valid_outputs': {'A': 8, 'B': 7, 'C': 8},
 'calls': 21,
 'additional_calls_on_completed_resume': 0,
 'total_call_latency_seconds': 0.00036220697802491486,
 'accepted_originals': 3,
 'revisions_requested': 5,
 'revisions_accepted': 2,
 'fallbacks': 3,
 'parser_failures': 1,
 'narrative_failures': 6,
 'inference_failures': 1,
 'checkpoint': '/private/var/folders/sb/xd5lskw56px2y86njz2clqpr0000gp/T/hrp-experiment2-fictional-75faeevz/checkpoint.json'}

## Verification trace

Inspect one failed revision and its deterministic output.

In [6]:
checkpoint = json.loads(Path(report["checkpoint"]).read_text())
records = list(checkpoint["records"].values())
fallback = next(row["verification"] for row in records
                if row["condition"] == "C" and row["verification"]["fallback_used"])
display(fallback)

{'original_raw': '{"explanation": "Sodium was 138 mEq/L. Sodium causes readmission.", "drivers": [{"feature": "sodium_missing", "direction": "negative"}, {"feature": "age", "direction": "positive"}, {"feature": "creatinine", "direction": "negative"}]}',
 'original': {'parse': {'raw': '{"explanation": "Sodium was 138 mEq/L. Sodium causes readmission.", "drivers": [{"feature": "sodium_missing", "direction": "negative"}, {"feature": "age", "direction": "positive"}, {"feature": "creatinine", "direction": "negative"}]}',
   'answer': {'explanation': 'Sodium was 138 mEq/L. Sodium causes readmission.',
    'drivers': [{'feature': 'sodium_missing', 'direction': 'negative'},
     {'feature': 'age', 'direction': 'positive'},
     {'feature': 'creatinine', 'direction': 'negative'}]},
   'error': None,
   'warnings': []},
  'structured': {'expected': [{'feature': 'sodium_missing',
     'direction': 'negative'},
    {'feature': 'age', 'direction': 'positive'},
    {'feature': 'creatinine', 'directi

## Fictional scoring

The three comparisons use the same attribution references. The paired
counts show which cases contributed to quality averages. Condition C
is also reported separately by its output source.

In [7]:
fictional_scores = score_experiment(fictional_cases(), checkpoint)
display({name: pair["pairs"] for name, pair in fictional_scores["paired"].items()})
display(fictional_scores["by_condition_and_source"]["C"])

{'A_to_B': 7, 'B_to_C': 7, 'A_to_C': 8}

{'grounded_draft': {'records': 3,
  'evaluated': 3,
  'statuses': {'ok': 3},
  'parse_failures': 0,
  'inference_failures': {},
  'calls': 0,
  'latency_seconds': 0.0,
  'quality_means': {'top_three_overlap': 2,
   'signed_top_three_agreement': 2,
   'rank_position_agreement': 2,
   'exact_top_three_ordering': 1,
   'missing_influential_drivers': 0,
   'incorrect_feature_directions': 0,
   'structured_correct': 1,
   'structured_complete': 1,
   'explanation_correct': 1,
   'explanation_complete': 1,
   'missing_prose_drivers': 0,
   'narrative_contradictions': 0,
   'unsupported_model_claims': 0,
   'narrative_claims_checked': 2,
   'narrative_claims_unparsed': 0},
  'original_verification_statuses': {'verified_complete': 3},
  'revision_statuses': {},
  'intervention_counts': {'draft_accepted': 3,
   'revision_requested': 0,
   'revision_accepted': 0,
   'revision_rejected': 0,
   'fallback_used': 0}},
 'model_revision': {'records': 2,
  'evaluated': 2,
  'statuses': {'ok': 2},
  'pa

## Real benchmark · explicitly enabled

Set `RUN_REAL_BENCHMARK = True` and choose a private output directory
outside Git to start or resume the pinned 72-case run. Reusing that
directory resumes its checkpoint. The planned maximum is 216 generation
requests, including revisions. A complete run has 216 condition records,
including terminal failures. If a runtime budget is reached, run this
cell again with the same directory and model configuration to resume.
Only aggregate scores are rendered here.

In [8]:
RUN_REAL_BENCHMARK = True
REAL_OUTPUT_DIRECTORY = "~/Library/Application Support/AI380H/MIMIC-IV"  # Set to an absolute private directory outside Git.
REAL_RUNTIME_SECONDS = 4 * 60 * 60

if RUN_REAL_BENCHMARK:
    if REAL_OUTPUT_DIRECTORY is None:
        raise ValueError("Choose a private output directory for the real benchmark.")
    real_output = Path(REAL_OUTPUT_DIRECTORY).expanduser()
    if not real_output.is_absolute():
        raise ValueError("Use an absolute path for the real benchmark checkpoint.")

    from experiment2_inputs import load_explanation_cases

    provenance, cases = load_explanation_cases()
    if len(cases) != 72:
        raise RuntimeError("The real benchmark requires the 72 pinned cases.")
    client = Ollama.discover()
    if not client.config.digest:
        raise RuntimeError("The real benchmark requires an installed model digest.")
    real_run = run_experiment(
        cases, client, real_output, provenance=provenance,
        max_calls=3 * len(cases), runtime_seconds=REAL_RUNTIME_SECONDS,
    )
    real_scores = score_experiment(cases, real_run)
    totals = real_scores["operations"]
    if totals["records"] != 216:
        raise RuntimeError("Expected 216 condition records for the pinned case set.")
    display({
        "complete": totals["statuses"].get("pending", 0) == 0,
        "pause_reason": real_run["pause_reason"],
        "operations": totals,
        "by_condition": real_scores["by_condition"],
        "by_condition_and_source": real_scores["by_condition_and_source"],
        "paired": real_scores["paired"],
    })

{'complete': True,
 'pause_reason': None,
 'operations': {'records': 216,
  'statuses': {'ok': 214, 'parse_error': 2},
  'parse_failures': 9,
  'inference_failures': {},
  'calls': 216,
  'latency_seconds': 2524.6465354620304,
  'intervention_counts': {'draft_accepted': 0,
   'revision_requested': 72,
   'revision_accepted': 8,
   'revision_rejected': 64,
   'fallback_used': 64}},
 'by_condition': {'A': {'records': 72,
   'evaluated': 70,
   'statuses': {'ok': 70, 'parse_error': 2},
   'parse_failures': 2,
   'inference_failures': {},
   'calls': 72,
   'latency_seconds': 802.7466170860571,
   'quality_means': {'top_three_overlap': 1.3428571428571427,
    'signed_top_three_agreement': 1,
    'rank_position_agreement': 0.38571428571428573,
    'exact_top_three_ordering': 0.014285714285714285,
    'missing_influential_drivers': 1.6571428571428573,
    'incorrect_feature_directions': 0.7142857142857143,
    'structured_correct': 0.07142857142857142,
    'structured_complete': 0.0428571428